# Phase 1: head-only baseline on PTB-XL (Yashwante & Yu, ICML 2026)
Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.
If Colab disconnects, Run all again: finished pieces are skipped.
Expected: ~1–1.5 h total, mostly waiting. Results land in `My Drive/rose-yu-ecg/results/`.

In [ ]:
# 1. GPU check: stop here if this fails
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then rerun."
print("CUDA OK:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Mount Drive and go to the project folder
import glob, os
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/rose-yu-ecg'
if not os.path.exists(PROJECT + '/src/extract.py'):  # folder was dropped somewhere else in Drive
    PROJECT = glob.glob('/content/drive/MyDrive/**/rose-yu-ecg/src/extract.py', recursive=True)[0].rsplit('/src/', 1)[0]
print(PROJECT)
%cd {PROJECT}
!ls

In [ ]:
# 3. Install pinned dependencies (torch comes with Colab)
!pip install -q -r requirements-colab.txt
!pip install -q momentfm==0.1.4 --no-deps

In [ ]:
# 4. Download PTB-XL 1.0.3 (500 Hz records + metadata) to fast local disk, SHA256-checked
!python src/download_ptbxl.py --out /content/ptbxl

In [ ]:
# 5. Frozen encoders -> cached embeddings on Drive (config A: MOMENT-B, DINOv2-B, Qwen3-Emb-0.6B)
#    Look for the "[cuda ok] <GPU name>" lines: they prove each encoder ran on the GPU.
!python src/extract.py --ptbxl /content/ptbxl --out results/emb --config A 2>&1 | tee -a results/extract_A.log

In [ ]:
# 6. Train projection heads (3 seeds) + untrained-heads floor, then compare with the paper
!python src/train_heads.py --emb results/emb --config A --seeds 0 1 2 2>&1 | tee results/train_A.log
!python src/train_heads.py --emb results/emb --config A --seeds 0 1 2 --epochs 0 2>&1 | tee results/train_A_untrained.log
!python src/compare.py --emb results/emb --out results/phase1_results.md

## Optional: config D (SigLIP2-B + Qwen3-Emb-0.6B + Chronos-B)
Only if there is time left. Qwen embeddings are reused from config A.

In [ ]:
RUN_CONFIG_D = False
if RUN_CONFIG_D:
    !python src/extract.py --ptbxl /content/ptbxl --out results/emb --config D 2>&1 | tee -a results/extract_D.log
    !python src/train_heads.py --emb results/emb --config D --seeds 0 1 2 2>&1 | tee results/train_D.log
    !python src/train_heads.py --emb results/emb --config D --seeds 0 1 2 --epochs 0 2>&1 | tee results/train_D_untrained.log
    !python src/compare.py --emb results/emb --out results/phase1_results.md

In [ ]:
# 7. Optional: lead II check (text embeddings are reused; only TS and IMG are re-extracted, ~45 min)
!test -d /content/ptbxl/records500 || python src/download_ptbxl.py --out /content/ptbxl
!python src/extract.py --ptbxl /content/ptbxl --out results/emb --config A --lead II 2>&1 | tee -a results/extract_A_leadII.log
!python src/train_heads.py --emb results/emb --config A --lead II --seeds 0 1 2 --out results/emb/results_A_leadII.json 2>&1 | tee results/train_A_leadII.log

In [ ]:
# 8. Label-level evaluation (diagnostic superclasses; zero-shot AUROC; same-diagnosis retrieval)
!test -f /content/ptbxl/scp_statements.csv || python src/download_ptbxl.py --out /content/ptbxl --only 0
!python src/label_eval.py --emb results/emb --ptbxl /content/ptbxl --config A --lead I 2>&1 | tee results/label_eval_A_leadI.log
!python src/label_eval.py --emb results/emb --ptbxl /content/ptbxl --config A --lead II 2>&1 | tee results/label_eval_A_leadII.log

In [ ]:
# 9. Phase-2 pilot: head vs. deeper head vs. LoRA at a matched trainable-parameter budget (lead I)
#    Checkpoints every epoch: if Colab disconnects, rerun this cell and it resumes.
!pip install -q peft==0.17.1
!test -d /content/ptbxl/records500 || test -f results/emb/signals_test_leadI.npy || python src/download_ptbxl.py --out /content/ptbxl
!python src/adapt.py --emb results/emb --ptbxl /content/ptbxl --arm head deep_head lora --seeds 0 --micro 128 2>&1 | tee -a results/adapt_leadI.log